In [1]:
# 데이터 처리 및 분석
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 통계 분석
from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, chi2_contingency, f_oneway
from scipy.stats import mannwhitneyu, fisher_exact, kruskal
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
import pingouin as pg
import scikit_posthocs as sp

# 머신러닝 
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report, r2_score, mean_squared_error, roc_auc_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
from xgboost import XGBClassifier, XGBRegressor
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer

# 출력 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 시드 설정
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [2]:
df = pd.read_csv('data/merged_final_data.csv')

In [3]:
cols = df.columns
cols 

Index(['order_id', 'customer_id', 'customer_unique_id', 'customer_city',
       'customer_state', 'order_item_id', 'seller_id', 'shipping_limit_date',
       'price', 'freight_value', 'review_score', 'category', 'seller_city',
       'seller_state', 'order_purchase_dayofweek', 'order_purchase_month',
       'approved_days', 'dispatch_days', 'delivery_days',
       'expected_delivery_days', 'delay_days', 'delay_days_int', 'is_delayed',
       'delay_days_cat', 'main_category', 'sub_category', 'distance_km',
       'cross_state'],
      dtype='str')

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 109294 entries, 0 to 109293
Data columns (total 28 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   order_id                  109294 non-null  str    
 1   customer_id               109294 non-null  str    
 2   customer_unique_id        109294 non-null  str    
 3   customer_city             109294 non-null  str    
 4   customer_state            109294 non-null  str    
 5   order_item_id             109294 non-null  int64  
 6   seller_id                 109294 non-null  str    
 7   shipping_limit_date       109294 non-null  str    
 8   price                     109294 non-null  float64
 9   freight_value             109294 non-null  float64
 10  review_score              109294 non-null  int64  
 11  category                  107749 non-null  str    
 12  seller_city               109294 non-null  str    
 13  seller_state              109294 non-null  str    
 14 

In [5]:
df.isna().sum()

order_id                       0
customer_id                    0
customer_unique_id             0
customer_city                  0
customer_state                 0
order_item_id                  0
seller_id                      0
shipping_limit_date            0
price                          0
freight_value                  0
review_score                   0
category                    1545
seller_city                    0
seller_state                   0
order_purchase_dayofweek       0
order_purchase_month           0
approved_days                  0
dispatch_days                  0
delivery_days                  0
expected_delivery_days         0
delay_days                     0
delay_days_int                 0
is_delayed                     0
delay_days_cat                 0
main_category                  0
sub_category                   0
distance_km                    0
cross_state                    0
dtype: int64

### 머신러닝 모델 생성시 절대 안쓸만한 컬럼 제거

In [6]:
cols_to_drop = ['order_id','customer_id','seller_id','category','delay_days_int']
df = df.drop(columns=cols_to_drop)
df['delay_days'] = df['delay_days'].astype(int) # delay_days 정수화

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 109294 entries, 0 to 109293
Data columns (total 23 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   customer_unique_id        109294 non-null  str    
 1   customer_city             109294 non-null  str    
 2   customer_state            109294 non-null  str    
 3   order_item_id             109294 non-null  int64  
 4   shipping_limit_date       109294 non-null  str    
 5   price                     109294 non-null  float64
 6   freight_value             109294 non-null  float64
 7   review_score              109294 non-null  int64  
 8   seller_city               109294 non-null  str    
 9   seller_state              109294 non-null  str    
 10  order_purchase_dayofweek  109294 non-null  str    
 11  order_purchase_month      109294 non-null  int64  
 12  approved_days             109294 non-null  int64  
 13  dispatch_days             109294 non-null  int64  
 14 

In [8]:
df.isna().sum()

customer_unique_id          0
customer_city               0
customer_state              0
order_item_id               0
shipping_limit_date         0
price                       0
freight_value               0
review_score                0
seller_city                 0
seller_state                0
order_purchase_dayofweek    0
order_purchase_month        0
approved_days               0
dispatch_days               0
delivery_days               0
expected_delivery_days      0
delay_days                  0
is_delayed                  0
delay_days_cat              0
main_category               0
sub_category                0
distance_km                 0
cross_state                 0
dtype: int64

In [9]:
df.to_csv('./data/ml_data.csv')

In [10]:
df2 = df.copy()
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 109294 entries, 0 to 109293
Data columns (total 23 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   customer_unique_id        109294 non-null  str    
 1   customer_city             109294 non-null  str    
 2   customer_state            109294 non-null  str    
 3   order_item_id             109294 non-null  int64  
 4   shipping_limit_date       109294 non-null  str    
 5   price                     109294 non-null  float64
 6   freight_value             109294 non-null  float64
 7   review_score              109294 non-null  int64  
 8   seller_city               109294 non-null  str    
 9   seller_state              109294 non-null  str    
 10  order_purchase_dayofweek  109294 non-null  str    
 11  order_purchase_month      109294 non-null  int64  
 12  approved_days             109294 non-null  int64  
 13  dispatch_days             109294 non-null  int64  
 14 

In [11]:
# feature engineering

df2 = df.copy()

# 베송속도 관련 파생피처들
df2['delivery_speed'] = df2['distance_km'] / df2['delivery_days']
df2['day_per_km'] = df2['delivery_days'] / df2['distance_km']
df2['delivery_ratio'] = df2['delivery_days'] / df2['expected_delivery_days']

# 가격 대비 배송비
df2['freight_ratio'] = df2['freight_value'] / df2['price']

# delivery_days * distance_km
df2["delivery_distance"] = df2["delivery_days"] * df2["distance_km"]

# delivery_days * price
df2["delivery_price"] = (df2["delivery_days"] * df2["price"])

# price + freight_value
df2['total_price'] = df2['price'] + df2['freight_value']

df2 = df2.replace([np.inf, -np.inf], np.nan)
num_cols = df2.select_dtypes(include=[np.number]).columns

In [12]:
# 상파울루 고객 여부 (True/False를 1/0으로 변환)
df2['is_sp_customer'] = (df2['customer_state'] == 'SP').astype(int)

# 기본값을 0으로 설정 (둘 다 SP가 아닌 경우: 일반/장거리 노선)
df2['sp_route_type'] = 0

# 조건 1: 한 쪽이라도 SP인 경우 (부분적 인프라 혜택)
cond_partial = (df2['seller_state'] == 'SP') | (df2['customer_state'] == 'SP')
df2.loc[cond_partial, 'sp_route_type'] = 1

# 조건 2: 둘 다 SP인 경우 (최적화된 인프라, 라스트마일 최적화)
cond_internal = (df2['seller_state'] == 'SP') & (df2['customer_state'] == 'SP')
df2.loc[cond_internal, 'sp_route_type'] = 2

In [13]:
# 브라질 물류 특성을 반영한 경계값 설정
# 0-50(도시내), 50-200(인접도시), 200-600(핵심노선), 600-1200(지역간), 1200+(초장거리)
bins = [0, 50, 250, 750, 1500, np.inf]
labels = [0, 1, 2, 3, 4]

df2['distance_cat'] = pd.cut(
    df2['distance_km'], 
    bins=bins, 
    labels=labels, 
    include_lowest=True
).astype(int)

# 각 구간에 데이터가 골고루 분포되었는지 확인
print(df2['distance_cat'].value_counts().sort_index())

distance_cat
0    13382
1    17689
2    48704
3    19790
4     9729
Name: count, dtype: int64


In [14]:
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 109294 entries, 0 to 109293
Data columns (total 33 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   customer_unique_id        109294 non-null  str    
 1   customer_city             109294 non-null  str    
 2   customer_state            109294 non-null  str    
 3   order_item_id             109294 non-null  int64  
 4   shipping_limit_date       109294 non-null  str    
 5   price                     109294 non-null  float64
 6   freight_value             109294 non-null  float64
 7   review_score              109294 non-null  int64  
 8   seller_city               109294 non-null  str    
 9   seller_state              109294 non-null  str    
 10  order_purchase_dayofweek  109294 non-null  str    
 11  order_purchase_month      109294 non-null  int64  
 12  approved_days             109294 non-null  int64  
 13  dispatch_days             109294 non-null  int64  
 14 

In [15]:
### 상파울루인지 구분하는 컬럼
# 기본값을 0으로 설정 (둘 다 SP가 아닌 경우: 일반/장거리 노선)
df2['sp_route_type'] = 0

# 조건 1: 한 쪽이라도 SP인 경우 (부분적 인프라 혜택)
cond_partial = (df2['seller_state'] == 'SP') | (df2['customer_state'] == 'SP')
df2.loc[cond_partial, 'sp_route_type'] = 1

# 조건 2: 둘 다 SP인 경우 (최적화된 인프라, 라스트마일 최적화)
cond_internal = (df2['seller_state'] == 'SP') & (df2['customer_state'] == 'SP')
df2.loc[cond_internal, 'sp_route_type'] = 2



### 상파울루 고객 여부 (True/False를 1/0으로 변환)
df2['is_sp_customer'] = (df2['customer_state'] == 'SP').astype(int)

# 기본값을 0으로 설정 (둘 다 SP가 아닌 경우: 일반/장거리 노선 및 고객이 SP가 아닌 경우)
df2['sp_route_type_customer'] = 0

# 조건 1: 고객이 SP인 경우 (부분적 인프라 혜택)
cond_partial = (df2['customer_state'] == 'SP')
df2.loc[cond_partial, 'sp_route_type_customer'] = 1

# 조건 2: 둘 다 SP인 경우 (최적화된 인프라, 라스트마일 최적화)
cond_internal = (df2['seller_state'] == 'SP') & (df2['customer_state'] == 'SP')
df2.loc[cond_internal, 'sp_route_type_customer'] = 2

# 결과 확인
print(df2['sp_route_type_customer'].value_counts().sort_index())



### 상파울루 셀러 여부 (True/False를 1/0으로 변환)
df2['is_sp_seller'] = (df2['seller_state'] == 'SP').astype(int)

# 기본값을 0으로 설정 (둘 다 SP가 아닌 경우: 일반/장거리 노선 및 셀러가 SP가 아닌 경우)
df2['sp_route_type_seller'] = 0

# 조건 1: 셀러가 SP인 경우 (부분적 인프라 혜택)
cond_partial_seller = (df2['seller_state'] == 'SP')
df2.loc[cond_partial_seller, 'sp_route_type_seller'] = 1

# 조건 2: 둘 다 SP인 경우 (최적화된 인프라, 라스트마일 최적화)
cond_internal_seller = (df2['seller_state'] == 'SP') & (df2['customer_state'] == 'SP')
df2.loc[cond_internal_seller, 'sp_route_type_seller'] = 2

# 결과 확인
print(df2['sp_route_type_seller'].value_counts().sort_index())

sp_route_type_customer
0    63171
1    10959
2    35164
Name: count, dtype: int64
sp_route_type_seller
0    31348
1    42782
2    35164
Name: count, dtype: int64


In [16]:
# 데이터셋 추출
df2.to_csv('./data/ml_data_fe.csv', index=False)
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 109294 entries, 0 to 109293
Data columns (total 36 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   customer_unique_id        109294 non-null  str    
 1   customer_city             109294 non-null  str    
 2   customer_state            109294 non-null  str    
 3   order_item_id             109294 non-null  int64  
 4   shipping_limit_date       109294 non-null  str    
 5   price                     109294 non-null  float64
 6   freight_value             109294 non-null  float64
 7   review_score              109294 non-null  int64  
 8   seller_city               109294 non-null  str    
 9   seller_state              109294 non-null  str    
 10  order_purchase_dayofweek  109294 non-null  str    
 11  order_purchase_month      109294 non-null  int64  
 12  approved_days             109294 non-null  int64  
 13  dispatch_days             109294 non-null  int64  
 14 

# 긍정 부정 그룹화 기준!

In [17]:
# 분류모델 시 그룹화 함수 정의
def group_score(score):
    if score >= 4: return 1   # High (긍정)
    else: return 0            # Low (부정)

X = df2.drop(columns=['review_score','customer_unique_id','customer_city','customer_state',
                      'shipping_limit_date','seller_city','seller_state'])
y = df2['review_score'].apply(group_score)

In [18]:
# 원핫인코딩 (자동으로 범주형만 변환, 수치형은 유지)
X = pd.get_dummies(X, drop_first=True)

In [19]:
# Train+Valid / Test 분리
X_train_valid, X_test, y_train_valid, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train / Valid 분리
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_valid, y_train_valid,
    test_size=0.2,
    random_state=42,
    stratify=y_train_valid
)

In [20]:
# 7. XGBoost 모델 설정 (M4 Pro 최적화)
xgb_model = XGBClassifier(
    n_estimators=1000,       # 조기 종료를 믿고 나무를 넉넉하게 1000개로 줍니다.
    max_depth=6,             # 너무 깊으면 과적합되므로 6~8 정도가 좋습니다.
    learning_rate=0.05,      # 꼼꼼하게 학습하기 위해 학습률을 낮춥니다.
    tree_method='hist',      # Mac CPU 가속
    random_state=42,
    n_jobs=-1,               # M4 Pro 모든 코어 사용
    early_stopping_rounds=50 # ★ 핵심: 30번 연속으로 Valid 점수가 안 오르면 자동 종료
)

# 8. 모델 학습 (Valid 세트를 활용해 실시간으로 평가)
print("모델 학습을 시작합니다...")
xgb_model.fit(
    X_train, 
    y_train,
    eval_set=[(X_valid, y_valid)], # Valid 세트를 알려줍니다.
    verbose=50                     # 50번 학습할 때마다 점수를 출력해줍니다.
)

# 9. 최종 테스트 세트(Test) 평가
# 조기 종료로 찾아낸 '가장 좋았던 때의 나무 개수(best_iteration)'로 예측합니다.
y_pred = xgb_model.predict(X_test)

print("\n[최종 평가 결과]")
print(f"최적의 나무 개수(Best Iteration): {xgb_model.best_iteration}")
print(f"XGBoost 최종 테스트 정확도: {accuracy_score(y_test, y_pred):.4f}")
print("\n상세 리포트:\n", classification_report(y_test, y_pred))

모델 학습을 시작합니다...
[0]	validation_0-logloss:0.53413
[50]	validation_0-logloss:0.47349
[100]	validation_0-logloss:0.47024
[150]	validation_0-logloss:0.46857
[200]	validation_0-logloss:0.46711
[250]	validation_0-logloss:0.46619
[300]	validation_0-logloss:0.46478
[350]	validation_0-logloss:0.46375
[400]	validation_0-logloss:0.46295
[450]	validation_0-logloss:0.46219
[500]	validation_0-logloss:0.46150
[550]	validation_0-logloss:0.46077
[600]	validation_0-logloss:0.46061
[650]	validation_0-logloss:0.46003
[700]	validation_0-logloss:0.45947
[750]	validation_0-logloss:0.45889
[800]	validation_0-logloss:0.45875
[850]	validation_0-logloss:0.45827
[900]	validation_0-logloss:0.45778
[950]	validation_0-logloss:0.45739
[999]	validation_0-logloss:0.45702

[최종 평가 결과]
최적의 나무 개수(Best Iteration): 992
XGBoost 최종 테스트 정확도: 0.8171

상세 리포트:
               precision    recall  f1-score   support

           0       0.79      0.29      0.42      5064
           1       0.82      0.98      0.89     16795

    accu

In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 데이터 로드
df = pd.read_csv('data/ml_data_fe.csv')

# 2. 불필요한 컬럼 제거 (ID, 시간, 타겟과 겹치는 변수 등)
drop_cols = [
    'customer_unique_id', 'customer_city', 'seller_city', 'shipping_limit_date'
]
df_processed = df.drop(columns=drop_cols, errors='ignore')

# 3. 범주형 변수 원핫 인코딩 
# (이전 분류 모델의 피처 중요도에 있던 형태(예: sub_category_Furniture & Decor)로 자동 변환)
df_encoded = pd.get_dummies(df_processed)

# 4. 독립변수(X)와 종속변수(y) 분리
X = df_encoded.drop(columns=['review_score'])
y = df_encoded['review_score']

# 5. 학습용/테스트용 데이터 분리 (8:2 비율)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# 5. XGBoost 회귀 모델 생성 및 학습
xgb_model = XGBRegressor(
    n_estimators=1300,        # 트리 개수 (M4 Pro 성능이 좋으므로 더 늘려도 무방함)
    learning_rate=0.04,      # 학습률
    max_depth=10,             # 트리의 최대 깊이
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',      # Apple Silicon (M4 Pro) 환경에서 빠른 연산 지원
    n_jobs=-1,               # 사용 가능한 모든 코어 사용
    random_state=42
)

# 모델 학습 진행
print("모델 학습을 시작합니다...")
xgb_model.fit(X_train, y_train)
print("학습 완료!")

# 7. 예측 및 성능 평가
y_pred_valid = xgb_model.predict(X_valid)
y_pred = xgb_model.predict(X_test)

# 리뷰 스코어는 1~5점 사이이므로, 예측값이 이 범위를 벗어나지 않도록 클리핑(Clipping) 처리
y_pred = np.clip(y_pred, 1, 5)
y_pred_valid = np.clip(y_pred_valid, 1, 5)

remse_valid = np.sqrt(mean_squared_error(y_valid, y_pred_valid))
print(f"Validation RMSE: {remse_valid:.4f}")
mae_valid = mean_absolute_error(y_valid, y_pred_valid)
print(f"Validation MAE: {mae_valid:.4f}")
r2_valid = r2_score(y_valid, y_pred_valid)
print(f"Validation R2 Score: {r2_valid:.4f}")

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n=== 모델 평가 지표 ===")
print(f"RMSE (평균 제곱근 오차): {rmse:.4f}")
print(f"MAE (평균 절대 오차): {mae:.4f}")
print(f"R2 Score (결정계수): {r2:.4f}")


모델 학습을 시작합니다...
학습 완료!
Validation RMSE: 1.1391
Validation MAE: 0.8508
Validation R2 Score: 0.2905

=== 모델 평가 지표 ===
RMSE (평균 제곱근 오차): 1.1301
MAE (평균 절대 오차): 0.8460
R2 Score (결정계수): 0.2990
